In [16]:
import numpy as np
import pandas as pd
import pickle
import argparse
import io, time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

In [17]:
def extract_features(df: pd.DataFrame) -> np.ndarray:
    """
    特征提取：仅 3 维，直接取原始值，无额外计算
 
    物理依据：最优 Tuner State = f(默认态阻抗 γ₂, 工作频点)
              与用户接触状态 γ₁ 无关（泛化核心）
    """
    d = df.copy()
    if '特征一实部' in d.columns:
        d.columns = ['gammaIn1Re','gammaIn1Im','closeFreqMHz',
                     'itState','gammaIn2Re','gammaIn2Im']
    return np.column_stack([
        d['gammaIn2Re'].values,
        d['gammaIn2Im'].values,
        d['closeFreqMHz'].values,
    ]).astype(np.float32)
 
 
def train(x_path: str, y_path: str, save_path: str) -> DecisionTreeClassifier:
    X_raw = pd.read_csv(x_path, encoding='gbk')
    X_raw.columns = ['gammaIn1Re','gammaIn1Im','closeFreqMHz',
                     'itState','gammaIn2Re','gammaIn2Im']
    y = pd.read_csv(y_path)['label']
 
    X = extract_features(X_raw)
 
    # 用全量数据训练
    model = DecisionTreeClassifier(
        max_depth=25,
        class_weight='balanced',
        random_state=42,
    )
    model.fit(X, y)
 
    # 5折交叉验证评估
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    from sklearn.model_selection import cross_val_score
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro')
    print(f'5-fold CV F1-macro: {scores.mean():.5f} ± {scores.std():.5f}')
    print(f'叶节点数: {model.get_n_leaves()}')
 
    # 延迟测试
    sample = X[:1]
    t0 = time.perf_counter()
    for _ in range(10000): model.predict(sample)
    ms = (time.perf_counter() - t0) / 10000 * 1000
    print(f'单样本延迟: {ms:.4f}ms  ({1000/ms:.0f} QPS)')
 
    buf = io.BytesIO(); pickle.dump(model, buf)
    print(f'模型大小: {buf.tell()/1024:.0f} KB')
 
    with open(save_path, 'wb') as f: pickle.dump(model, f)
    print(f'已保存: {save_path}')
    return model
 
 
class AntennaStatePredictor:
    """在线推理器"""
    def __init__(self, model_path='antenna_model_final.pkl'):
        with open(model_path, 'rb') as f:
            self._model = pickle.load(f)
 
    def predict(self, gammaIn1Re, gammaIn1Im, closeFreqMHz,
                itState, gammaIn2Re, gammaIn2Im) -> int:
        """单样本推理，返回最优 Tuner State (0~14)"""
        x = np.array([[gammaIn2Re, gammaIn2Im, closeFreqMHz]], dtype=np.float32)
        return int(self._model.predict(x)[0])
 
    def predict_batch(self, df: pd.DataFrame) -> np.ndarray:
        """批量推理"""
        return self._model.predict(extract_features(df))
 
    def benchmark(self, n=10000):
        import time
        dummy = np.zeros((1, 3), dtype=np.float32)
        t0 = time.perf_counter()
        for _ in range(n): self._model.predict(dummy)
        ms = (time.perf_counter()-t0)/n*1000
        print(f'单样本延迟: {ms:.4f}ms  ({1000/ms:.0f} QPS)')

In [18]:
parser = argparse.ArgumentParser(description='IAT 天线状态分类')
parser.add_argument('--mode', choices=['train', 'infer', 'benchmark'], default='train')
parser.add_argument('--x_train', default=r'X:\test\26研电赛-IAT-公开数据集\x_train.csv')
parser.add_argument('--y_train', default=r'X:\test\26研电赛-IAT-公开数据集\y_train.csv')
parser.add_argument('--model',   default='antenna_model.pkl')
args, _ = parser.parse_known_args()

if args.mode == 'train':
    train(args.x_train, args.y_train, args.model)

elif args.mode == 'infer':
    predictor = AntennaStatePredictor(args.model)
    result = predictor.predict(
        gammaIn1Re=-1332, gammaIn1Im=20179,
        closeFreqMHz=846, itState=4,
        gammaIn2Re=-6596, gammaIn2Im=1067,
    )
    print(f'预测 Tuner State: {result}')

elif args.mode == 'benchmark':
    predictor = AntennaStatePredictor(args.model)
    predictor.benchmark(n=5000)

5-fold CV F1-macro: 1.00000 ± 0.00000
叶节点数: 467
单样本延迟: 0.0837ms  (11953 QPS)
模型大小: 169 KB
已保存: antenna_model.pkl
